# LLM-from-Scratch (124M Parameters)

Train a GPT-style Transformer from scratch on any Jupyter platform (Kaggle, SageMaker, local GPU, etc.).

**Features:**
- **~124M parameters** decoder-only Transformer
- **tiktoken (GPT-2)** tokenizer — battle-tested BPE
- **FineWeb-Edu** dataset — high-quality web text
- **Perplexity tracking** + sample generation during training
- **Session-safe** with automatic checkpoint resume from local storage
- **torch.compile** for ~1.5× speedup on CUDA

> **Tip:** Ensure you have a GPU enabled before starting.

## 1. Verify GPU & Install Dependencies

In [ ]:
!nvidia-smi
!pip install -q torch numpy tqdm requests matplotlib regex datasets tiktoken huggingface-hub

import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Clone Repository & Setup Directories

> Fetches the latest code from GitHub and creates local project folders.
> If you already have the repo cloned, you can skip the `git clone` line and just `cd` into it.

In [ ]:
import os

# Clone repository (skip if already present)
if not os.path.exists("llm-from-scratch"):
    !git clone https://github.com/avneeshjadhav04/llm-from-scratch.git

%cd llm-from-scratch

# Create local project directories
os.makedirs("checkpoints", exist_ok=True)
os.makedirs("logs", exist_ok=True)
os.makedirs("data", exist_ok=True)

print("Project ready. Directories created:")
print("  checkpoints/  — model checkpoints")
print("  logs/         — training logs & plots")
print("  data/         — tokenized dataset binaries")

## 3. Prepare Data (FineWeb-Edu)

> Configure the number of tokens and train/val split below.
> Default: 2B tokens (95/5 train/val split).
> **Takes ~20–40 minutes on first run.** Subsequent runs skip if binaries exist.

In [ ]:
# --- Data Preparation Configuration ---
NUM_TOKENS = 2_000_000_000   # Total tokens to download & tokenize (2B)
TRAIN_SPLIT = 0.95           # Train/validation split ratio (0.0–1.0)

print(f"Data prep config:")
print(f"  Target tokens:   {NUM_TOKENS:,}")
print(f"  Train split:     {TRAIN_SPLIT}")

!python prepare_data.py \
    --num_tokens {NUM_TOKENS} \
    --train_split {TRAIN_SPLIT}

## 4. Resume Setup — Check Local Checkpoints

> Checks local `checkpoints/` for existing models.
> Run this before every training session (fresh or resume).

In [ ]:
import os
import glob

os.makedirs("checkpoints", exist_ok=True)
os.makedirs("logs", exist_ok=True)

local_checkpoints = sorted(glob.glob("checkpoints/*.pt"))
if local_checkpoints:
    latest = local_checkpoints[-1]
    step = int(os.path.basename(latest).split("_")[-1].replace(".pt", ""))
    print(f"Found local checkpoint: {os.path.basename(latest)} (step {step})")
    print(f"Ready to resume from step {step}")
else:
    print("No local checkpoint found. Training will start from scratch.")

## 5. Train the 124M Model

> Adjust the hyperparameters below, then run the cell.
> Training will resume automatically if a checkpoint exists locally.

In [ ]:
# --- Training Configuration ---
# Adjust these values before each training session

BATCH_SIZE = 8               # Safe for most GPUs; increase to 16 on A100/T4 if you have VRAM headroom
LEARNING_RATE = 6e-4         # Default: 6e-4
SESSION_STEPS = 0            # 0 = train to completion in one go; set N for session capping
MAX_SEQ_LEN = 512            # Context length (no need to re-prepare data if changed)
WARMUP_STEPS = 2000          # LR warmup steps
USE_COMPILE = True           # torch.compile for speed (set False if unstable)
MAX_STEPS = 0                # 0 = auto-compute from NUM_TOKENS (set in Step 3)

# Reuse NUM_TOKENS from Step 3 if available; otherwise use config default
try:
    _nt = NUM_TOKENS
except NameError:
    _nt = 2_000_000_000

_grad_accum = 4  # must match config.grad_accum_steps
tokens_per_step = BATCH_SIZE * MAX_SEQ_LEN * _grad_accum
computed_steps = _nt // tokens_per_step

print(f"Training config:")
print(f"  Batch size:      {BATCH_SIZE}")
print(f"  Learning rate:   {LEARNING_RATE}")
print(f"  Session steps:   {SESSION_STEPS}")
print(f"  Max seq length:  {MAX_SEQ_LEN}")
print(f"  Warmup steps:    {WARMUP_STEPS}")
print(f"  torch.compile:   {USE_COMPILE}")
print(f"  Target tokens:   {_nt:,}")
print(f"  Tokens/step:     {tokens_per_step:,}")
print(f"  Computed steps:  {computed_steps:,}")
print(f"  Max steps:       {'auto' if MAX_STEPS == 0 else MAX_STEPS}")

!python train.py \
    --batch_size {BATCH_SIZE} \
    --learning_rate {LEARNING_RATE} \
    --max_steps_per_session {SESSION_STEPS} \
    --max_seq_len {MAX_SEQ_LEN} \
    --warmup_steps {WARMUP_STEPS} \
    --compile {USE_COMPILE} \
    --num_tokens {_nt} \
    --max_steps {MAX_STEPS}

## 6. Verify Local Checkpoints & Logs

> Lists all saved checkpoints and logs in local storage.
> Copy the `checkpoints/` and `logs/` folders manually if you want to back them up.

In [ ]:
import os
import re
import glob

# Leave as 0 to list all checkpoints, or enter a specific step to verify
SHOW_UP_TO_STEP = 0   # e.g., 5000

local_ckpts_raw = sorted(glob.glob("checkpoints/*.pt"))
step_ckpts = [c for c in local_ckpts_raw if re.search(r'step_\d+\.pt$', os.path.basename(c))]
best_ckpt = os.path.join("checkpoints", "100m_best.pt")

if not step_ckpts:
    print("No local step checkpoints found.")
else:
    print(f"Found {len(step_ckpts)} step checkpoint(s):")
    for ckpt in step_ckpts:
        match = re.search(r'step_(\d+)\.pt$', os.path.basename(ckpt))
        step = int(match.group(1))
        size_mb = os.path.getsize(ckpt) / (1024 * 1024)
        marker = " <-- latest" if ckpt == step_ckpts[-1] else ""
        if SHOW_UP_TO_STEP == 0 or step <= SHOW_UP_TO_STEP:
            print(f"  Step {step:6d}  ({size_mb:.1f} MB)  {os.path.basename(ckpt)}{marker}")

    total_size_gb = sum(os.path.getsize(c) for c in step_ckpts) / (1024 ** 3)
    print(f"\nTotal step checkpoint size: {total_size_gb:.2f} GB")

if os.path.exists(best_ckpt):
    size_mb = os.path.getsize(best_ckpt) / (1024 * 1024)
    print(f"\nBest checkpoint: 100m_best.pt  ({size_mb:.1f} MB)")

# List logs
log_files = glob.glob("logs/*")
if log_files:
    print("\nLog files:")
    for f in log_files:
        size_kb = os.path.getsize(f) / 1024
        print(f"  {os.path.basename(f)}  ({size_kb:.1f} KB)")
else:
    print("\nNo log files found.")

print("\nTip: Copy the 'checkpoints/' and 'logs/' folders to back up your work.")

## 7. Generate Text (Interactive)

> Uses the **latest local checkpoint** automatically.

In [ ]:
!python generate.py \
    --prompt "The future of artificial intelligence is" \
    --temperature 0.8 \
    --top_k 40 \
    --top_p 0.95 \
    --max_new_tokens 256 \
    --max_seq_len {MAX_SEQ_LEN} \
    --device cuda

## 8. Plot Training Curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("logs/100m_training_log.csv")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(df["step"], df["train_loss"], label="Train Loss")
val_mask = df["val_loss"].notna()
axes[0].plot(df.loc[val_mask, "step"], df.loc[val_mask, "val_loss"], label="Val Loss", linestyle="--")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training & Validation Loss")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(df["step"], df["train_ppl"], label="Train PPL")
axes[1].plot(df.loc[val_mask, "step"], df.loc[val_mask, "val_ppl"], label="Val PPL", linestyle="--")
axes[1].set_xlabel("Step")
axes[1].set_ylabel("Perplexity")
axes[1].set_title("Training & Validation Perplexity")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()